# Parsing a CSV File with Standard Python

In this demo we work with `reviews.csv`, a file of video game user reviews. We use only the **standard library** (no pandas):

1. Opening the CSV file
2. Checking the column names within the CSV file
3. Storing records and displaying information from the CSV file

> ⚠️ The `review_text` column contains real, unmoderated user reviews. Some are offensive, so check any text before you show it on screen.

In [4]:
import csv
from collections import Counter

FILE_PATH = "reviews.csv"

---
## Step 1: Opening the CSV file

We use `open()` inside a `with` block, so the file is **closed automatically** when the block ends, even if an error happens.

Two arguments matter for CSV files:

| Argument | Why |
|---|---|
| `encoding="utf-8"` | The reviews contain characters like `’`, `é`, `ñ`. Without it, Windows may use a different default encoding and crash or garble the text. |
| `newline=""` | Lets the `csv` module handle line endings itself. This matters here because some reviews **contain line breaks inside the text**. |

In [5]:
with open(FILE_PATH, encoding="utf-8", newline="") as file:
    # A file object is iterable: each loop gives us one raw line of text
    for line_number, line in enumerate(file):
        print(line_number, repr(line))
        if line_number == 4:
            break

0 'platform,review_id,published_date,score,review_text\r\n'
1 'playstation-5,mc-profile-73221238ba69e82808a52ea6f11af92abf38385594313fd8632f82e500efa45f,2026-11-21,6,"I guess it’s just more GTA.\n'
2 'No innovations, nothing extraordinary, just more GTA.\n'
3 'Probably better than GTA V? Hard to tell from what I’ve seen.\n'
4 'At least we are no longer in the PS3 era, that GTA V was."\r\n'


Look at lines 1 to 4: one review is spread across several raw lines. **A line is not the same as a record.**

So splitting each line on `,` would break. Commas can appear inside the review text, and so can newlines. Let's prove it by counting:

In [6]:
with open(FILE_PATH, encoding="utf-8", newline="") as file:
    raw_line_count = sum(1 for line in file)

print(f"Raw lines in the file: {raw_line_count:,}")

Raw lines in the file: 96,889


Now let's let `csv.reader` do the parsing. It understands quoted fields, so a value wrapped in `"..."` can safely contain commas and newlines.

`csv.reader` gives us each record as a **list of strings**.

In [3]:
with open(FILE_PATH, encoding="utf-8", newline="") as file:
    reader = csv.reader(file)

    for row_number, row in enumerate(reader):
        print(row_number, row)
        if row_number == 2:
            break

0 ['platform', 'review_id', 'published_date', 'score', 'review_text']
1 ['playstation-5', 'mc-profile-73221238ba69e82808a52ea6f11af92abf38385594313fd8632f82e500efa45f', '2026-11-21', '6', 'I guess it’s just more GTA.\nNo innovations, nothing extraordinary, just more GTA.\nProbably better than GTA V? Hard to tell from what I’ve seen.\nAt least we are no longer in the PS3 era, that GTA V was.']
2 ['playstation-5', 'mc-profile-24d933ed22bfa70a548d54edf921231c0883aa626c15ffda32961c62ab458a37', '2026-11-21', '0', 'Me decepciono, tanto tiempo que esperamos para que sea básicamente lo mismo y lleno de bugs, no se pueden hacer cosas que antes en el 5 si se podían, si agregaron una que otra cosa pero no vale la pena incluso retrasarlo para darnos esto, lo siento me decepcionaste gta VI']


In [ ]:
with open(FILE_PATH, encoding="utf-8", newline="") as file:
    reader = csv.reader(file)
    record_count = sum(1 for row in reader)

print(f"Raw lines:   {raw_line_count:,}")
print(f"CSV records: {record_count:,}  (including the header row)")

---
## Step 2: Checking the column names within the CSV file

The first row of the file is the **header**. It holds the column names. There are two ways to read it.

**Option A:** with `csv.reader`, call `next()` to take the first row.

In [ ]:
with open(FILE_PATH, encoding="utf-8", newline="") as file:
    reader = csv.reader(file)
    header = next(reader)   # take the first row only

print(header)
print(f"Number of columns: {len(header)}")

**Option B:** use `csv.DictReader`. It reads the header for us and exposes it as `.fieldnames`.

In [ ]:
with open(FILE_PATH, encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    column_names = reader.fieldnames

for index, name in enumerate(column_names, start=1):
    print(f"{index}. {name}")

Before relying on a column, check that it exists. This guards against a renamed or missing column.

In [ ]:
required_columns = ["platform", "score", "review_text"]

for column in required_columns:
    if column in column_names:
        print(f"✔ '{column}' found")
    else:
        print(f"✘ '{column}' is MISSING")

---
## Step 3: Storing records and displaying information from the CSV file

### 3a. Store the records

`csv.DictReader` gives each row as a **dictionary**: `{column_name: value}`. That reads much better than `row[3]`.

We copy the rows into a list **inside** the `with` block, because the reader can't read once the file is closed.

In [ ]:
reviews = []

with open(FILE_PATH, encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        reviews.append(row)

print(f"Stored {len(reviews):,} reviews")
print(type(reviews), type(reviews[0]))

### 3b. Look at a single record

In [ ]:
first_review = reviews[0]

for key, value in first_review.items():
    print(f"{key:>15}: {value}")

⚠️ **Every value from a CSV file is a string**, numbers included:

In [ ]:
print(repr(first_review["score"]), type(first_review["score"]))

# This is string comparison, not number comparison:
print("10" > "9")

### 3c. Clean the data while storing it

Let's read the file again. This time we convert `score` to an `int` as we go, and keep only the fields we need.

In [ ]:
reviews = []

with open(FILE_PATH, encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        reviews.append({
            "platform": row["platform"],
            "date": row["published_date"],
            "score": int(row["score"]),   # string -> int
            "text": row["review_text"].strip(),
        })

print(reviews[0]["score"], type(reviews[0]["score"]))

### 3d. Display the first few records

We print a short preview of each review so the output stays readable.

In [ ]:
def preview(text, length=70):
    """Return the text on one line, cut to `length` characters."""
    one_line = " ".join(text.split())
    return one_line if len(one_line) <= length else one_line[:length] + "..."


print(f"{'#':<3} {'PLATFORM':<15} {'DATE':<11} {'SCORE':>5}  REVIEW")
print("-" * 110)

for number, review in enumerate(reviews[:3], start=1):
    print(f"{number:<3} {review['platform']:<15} {review['date']:<11} {review['score']:>5}  {preview(review['text'])}")

### 3e. Answer questions about the data

**What is the average score?**

In [ ]:
total = 0
for review in reviews:
    total += review["score"]

average = total / len(reviews)
print(f"Average score: {average:.2f} / 10")

**How many reviews per platform?** The first version counts with a plain dictionary:

In [ ]:
platform_counts = {}

for review in reviews:
    platform = review["platform"]
    platform_counts[platform] = platform_counts.get(platform, 0) + 1

print(platform_counts)

`collections.Counter` from the standard library does the same counting in one line:

In [ ]:
platform_counts = Counter(review["platform"] for review in reviews)

for platform, count in platform_counts.most_common():
    print(f"{platform:<15} {count:>7,}")

**What is the average score per platform?**

In [ ]:
for platform in platform_counts:
    scores = [review["score"] for review in reviews if review["platform"] == platform]
    print(f"{platform:<15} {sum(scores) / len(scores):.2f}")

**How are the scores distributed?** A simple text bar chart:

In [ ]:
score_counts = Counter(review["score"] for review in reviews)

for score in range(0, 11):
    count = score_counts[score]
    bar = "█" * (count // 400)
    print(f"{score:>2} | {bar} {count:,}")

**How many reviews were published each day?**

In [ ]:
date_counts = Counter(review["date"] for review in reviews)

for date in sorted(date_counts):
    print(f"{date}: {date_counts[date]:,}")

**Filtering:** How many reviews gave a perfect 10 or a 0?

In [ ]:
perfect = [review for review in reviews if review["score"] == 10]
zero = [review for review in reviews if review["score"] == 0]

print(f"Perfect 10s: {len(perfect):,} ({len(perfect) / len(reviews):.1%})")
print(f"Zeros:       {len(zero):,} ({len(zero) / len(reviews):.1%})")

### 3f. A data-quality check: duplicate IDs

`review_id` *should* be unique. Let's check that:

In [ ]:
with open(FILE_PATH, encoding="utf-8", newline="") as file:
    ids = [row["review_id"] for row in csv.DictReader(file)]

unique_ids = set(ids)
print(f"Rows:       {len(ids):,}")
print(f"Unique IDs: {len(unique_ids):,}")
print(f"Duplicates: {len(ids) - len(unique_ids):,}")

Real data is messy, so always check your assumptions before you trust the numbers.

---
## Recap

| Step | Tool | Key idea |
|---|---|---|
| 1. Open | `with open(path, encoding="utf-8", newline="")` | The file closes automatically. A raw line is **not** a record. |
| 2. Columns | `next(csv.reader(f))` or `csv.DictReader(f).fieldnames` | The header row holds the column names. |
| 3. Store & display | `csv.DictReader` → list of dicts | Every value is a string, so convert types yourself. |

**Try it yourself:**
1. Find the longest review (by number of characters) and print its platform and score.
2. Count how many reviews on each platform scored 5 or less.
3. Write the perfect-10 reviews to a new file, `perfect_reviews.csv`, using `csv.DictWriter`.